# Tutorial: LD Clumping and Identifying Independent Signals with `cellink`

This tutorial demonstrates how to perform linkage disequilibrium (LD) clumping to identify independent genetic signals from QTL analysis results. LD clumping is essential for distinguishing truly independent associations from those driven by correlation between nearby variants due to linkage disequilibrium.
When performing genome-wide QTL mapping, you often identify multiple significant variants for the same gene or phenotype. Many of these variants may be in high LD with each other, meaning they tag the same underlying causal variant. Clumping helps identify "index" or "lead" variants that represent independent signals, which is crucial for reporting interpretable results with non-redundant associations, fine-mapping causal variants and prioritizing variants for functional validation.

This tutorial shows two complementary approaches:

- Post-QTL clumping: Clump significant results after QTL analysis
- Pre-analysis LD pruning: Reduce variant sets before analysis using reference LD

We'll use PLINK for clumping operations and demonstrate integration with `cellink` data structures. This tutorial assumes you've already performed eQTL analysis following earlier tutorials.

## Environment Setup
We begin by importing necessary libraries and defining parameters for clumping analysis.

In [1]:
import numpy as np
import pandas as pd
import subprocess
from pathlib import Path

import scanpy as sc
from cellink.resources import get_dummy_onek1k
from cellink.utils import column_normalize, gaussianize
from cellink.io import to_plink

chrom = 22
cis_window = 500_000
cell_type = "CD8 Naive"
celltype_key = "predicted.celltype.l2"
pb_gex_key = f"PB_{cell_type}"
original_donor_col = "donor_id"

# Clumping parameters
clump_p1 = 5e-8  # Significance threshold for index variants
clump_p2 = 1e-5  # Secondary significance threshold
clump_r2 = 0.1  # LD threshold
clump_kb = 500  # Physical distance window (kb)

site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [2]:
hd = get_dummy_onek1k(verify_checksum=False)
hd

[2026-11-23 23:00:08,729] INFO:root: ~/cellink_data/dummy_onek1k/dummy_onek1k_dataset.hd.h5 already exists


[2026-11-23 23:00:08,730] WARNING:root: No checksum provided, skipping verification


[2026-11-23 23:00:16,230] INFO:root: Loaded dummy OneK1K dataset: (300, 146939, 125913, 34830)


╔═ HiData(n_groups=300, n_cells=125,913) ══════════════════════════════════════╗
║ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓ ║
║ ┃ G (groups)                         ┃ C (cells, 333..420 per group)       ┃ ║
║ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩ ║
║ │ AnnData object with n_obs × n_vars │ AnnData object with n_obs × n_vars  │ ║
║ │ = 300 × 146,939                    │ = 125,913 × 34,830                  │ ║
║ │     var: 'AF', 'AN', 'ER2',        │     obs: 'orig.ident',              │ ║
║ │ 'IMPUTED', 'maf', 'NS', 'R2',      │ 'nCount_RNA', 'nFeature_RNA',       │ ║
║ │ 'TYPED', 'TYPED_ONLY', 'id',       │ 'percent.mt', 'donor_id',           │ ║
║ │ 'id_mask', 'length', 'pos',        │ 'pool_number',                      │ ║
║ │ 'quality', 'a0', 'a1', 'chrom',    │ 'predicted.celltype.l2',            │ ║
║ │ 'pos_hg19', 'id_hg19'              │ 'predicted.celltype.l2.score',      │ ║
║ │                                    │ 'age', 'organism_ontology_term_id', │ ║
║ │                                    │ 'tissue_ontology_term_id',          │ ║
║ │                                    │ 'assay_ontology_term_id',           │ ║
║ │                                    │ 'disease_ontology_term_id',         │ ║
║ │                                    │ 'cell_type_ontology_term_id',       │ ║
║ │                                    │ 'self_reported_ethnicity_ontology_… │ ║
║ │                                    │ 'development_stage_ontology_term_i… │ ║
║ │                                    │ 'sex_ontology_term_id',             │ ║
║ │                                    │ 'is_primary_data',                  │ ║
║ │                                    │ 'suspension_type', 'tissue_type',   │ ║
║ │                                    │ 'cell_type', 'assay', 'disease',    │ ║
║ │                                    │ 'organism', 'sex', 'tissue',        │ ║
║ │                                    │ 'self_reported_ethnicity',          │ ║
║ │                                    │ 'development_stage',                │ ║
║ │                                    │ 'observation_joinid'                │ ║
║ │     uns: 'dummy_onek1k', 'kinship' │     var: 'vst.mean',                │ ║
║ │                                    │ 'vst.variance',                     │ ║
║ │                                    │ 'vst.variance.expected',            │ ║
║ │                                    │ 'vst.variance.standardized',        │ ║
║ │                                    │ 'vst.variable',                     │ ║
║ │                                    │ 'feature_is_filtered',              │ ║
║ │                                    │ 'feature_name',                     │ ║
║ │                                    │ 'feature_reference',                │ ║
║ │                                    │ 'feature_biotype',                  │ ║
║ │                                    │ 'feature_length', 'feature_type',   │ ║
║ │                                    │ 'start', 'end', 'chrom', 'strand'   │ ║
║ │     obsm: 'gPCs'                   │     uns:                            │ ║
║ │                                    │ 'cell_type_ontology_term_id_colors… │ ║
║ │                                    │ 'citation', 'default_embedding',    │ ║
║ │                                    │ 'schema_reference',                 │ ║
║ │                                    │ 'schema_version', 'title'           │ ║
║ │                                    │     obsm: 'X_azimuth_spca',         │ ║
║ │                                    │ 'X_azimuth_umap', 'X_harmony',      │ ║
║ │                                    │ 'X_pca', 'X_umap'                   │ ║
║ │                                    │     varm: 'PCs'                     │ ║
║ └────────────────────────────────────┴─────────────────────────────────────┘ ║
╚═ group_id='donor_id' ════════════════════════════════════════════════════════╝

## Load Data and Perform QTL Analysis
First, we'll load the OneK1K dataset and perform a basic eQTL analysis to generate results for clumping. If you've already run eQTL analysis from Tutorial 1, you can skip this section and load your existing results.

In [3]:
hd.G.obs["donor_id"] = hd.G.obs.index

sc.pp.normalize_total(hd.C)
sc.pp.log1p(hd.C)

hd = hd[..., hd.C.obs[celltype_key] == cell_type, :].copy()
hd = hd.sel(G_var=hd.G.var.chrom == str(chrom), C_var=hd.C.var.chrom == str(chrom)).copy()

hd.aggregate(key_added=pb_gex_key, sync_var=True, verbose=True)
hd.aggregate(obs=["sex", "age"], func="first", add_to_obs=True)

print(f"Dataset shape: {hd.shape}")

[2026-11-23 23:00:35,941] INFO:hidata._core.hidata: Aggregated X to PB_CD8 Naive, observed for 294 groups.


Dataset shape: (294, 136776, 4859, 888)


## Quick eQTL Analysis
We'll run a simplified eQTL analysis to generate association statistics for clumping demonstration.

In [4]:
from cellink.at.gwas import GWAS
from tqdm.auto import tqdm

F = np.concatenate(
    [
        np.ones((hd.shape[0], 1)),
        hd.G.obs[["sex"]].values - 1,
        hd.G.obs[["age"]].values,
        hd.G.obsm["gPCs"].values[:, :10],
    ],
    axis=1,
)
F[:, 2:] = column_normalize(F[:, 2:])

if hasattr(hd.G.X, "compute"):
    hd.G.X = hd.G.X.compute()

# Test the genes expressed (non-zero pseudobulk) in at least half of the donors
expressed = (hd.G.obsm[pb_gex_key] > 0).mean(axis=0) >= 0.5
genes = hd.C.var.loc[expressed.index[expressed]]
print(f"Testing {len(genes)} of {hd.C.n_vars} genes")

rng = np.random.default_rng(0)  # seeded jitter, so tied expression values rank the same in every run
results = []
for gene, row in tqdm(list(genes.iterrows()), desc="Running eQTL tests"):
    Y = gaussianize(hd.G.obsm[pb_gex_key][[gene]].values + 1e-5 * rng.standard_normal((hd.shape[0], 1)))

    start = max(0, row.start - cis_window)
    end = row.end + cis_window
    _G = hd.G[:, (hd.G.var.pos >= start) & (hd.G.var.pos <= end)]
    _G = _G[:, (_G.X.std(0) != 0)]

    if _G.shape[1] == 0:
        continue

    gwas = GWAS(Y, F)
    gwas.test_association(_G.X)

    for i, snp_id in enumerate(_G.var.index):
        results.append(
            {
                "SNP": snp_id,
                "CHR": chrom,
                "BP": _G.var.iloc[i]["pos"],
                "GENE": gene,
                "BETA": gwas.getBetaSNP().ravel()[i],
                "P": gwas.getPv().ravel()[i],
            }
        )

eqtl_results = pd.DataFrame(results)
print(f"Total associations tested: {len(eqtl_results)}")
print(f"Significant associations (P < 1e-2): {(eqtl_results['P'] < 1e-2).sum()}")
eqtl_results.head()

Testing 98 of 888 genes


Running eQTL tests:   0%|          | 0/98 [00:00<?, ?it/s]

Total associations tested: 379508
Significant associations (P < 1e-2): 6834


,SNP,CHR,BP,GENE,BETA,P
0,22_16641442_T_C,22,16641442,ENSG00000069998,0.010501,0.926677
1,22_16642285_T_C,22,16642285,ENSG00000069998,0.010501,0.926677
2,22_16646267_C_G,22,16646267,ENSG00000069998,0.010890,0.923870
3,22_16647551_G_A,22,16647551,ENSG00000069998,0.082112,0.768991
4,22_16648018_A_G,22,16648018,ENSG00000069998,-0.019091,0.871392


## Approach 1: Post-QTL Clumping with PLINK
After obtaining QTL results, we identify independent signals by clumping variants based on LD patterns in our data. This approach uses your actual study genotypes to compute LD.
Step 1: Export Genotype Data to PLINK Format
PLINK requires binary format files (.bed, .bim, .fam). We'll export our genotype data from the HiData object.

In [5]:
plink_prefix = "chr22_genotypes"
to_plink(hd.G, plink_prefix)

[2026-11-23 23:01:22,113] INFO:cellink.io._export: Donor sex coded 0/1 is read as 0 = female, 1 = male (as in get_onek1k) and written as PLINK codes 2 (female) and 1 (male). Pass `donor_sex_map={0: 0, 1: 1}` if 0 means unknown.


[2026-11-23 23:01:24,114] INFO:cellink.io._export: Wrote PLINK 1 binary files chr22_genotypes.bed/.bim/.fam (294 x 136776)


### Step 1: Prepare Summary Statistics for Clumping
PLINK clumping requires summary statistics in a specific format. We'll prepare our eQTL results accordingly.

In [6]:
def prepare_clump_input(eqtl_results, output_file):
    """
    Prepare summary statistics for PLINK clumping.

    Parameters
    ----------
    eqtl_results : pd.DataFrame
        DataFrame with columns: SNP, CHR, BP, P (and optionally GENE, BETA)
    output_file : str
        Path to output file
    """
    clump_input = eqtl_results[["SNP", "CHR", "BP", "P"]].copy()
    clump_input = clump_input.sort_values("P")

    clump_input.to_csv(output_file, sep="\t", index=False)
    print(f"Prepared {len(clump_input)} associations for clumping")
    print(f"Saved to: {output_file}")

    return output_file


sumstats_file = prepare_clump_input(eqtl_results, "eqtl_sumstats.txt")

Prepared 379508 associations for clumping
Saved to: eqtl_sumstats.txt


### Step 2: Run PLINK Clumping
Now we'll run PLINK's clumping algorithm to identify independent signals. Here we perform clumping on the original genotype data. When your study sample is small or you want to use standard reference LD patterns, you can perform clumping using external genotype data, such as the 1000 Genomes Project data. This is particularly useful for comparing results across studies. The `cellink` package provides convenient access to 1000 Genomes data via the function `cellink.resources.get_1000genomes`.

In [7]:
def run_plink_clump(
    bfile, sumstats_file, output_prefix, clump_p1=0.00000005, clump_p2=0.00001, clump_r2=0.1, clump_kb=500
):
    """
    Run PLINK clumping to identify independent signals.

    Parameters
    ----------
    bfile : str
        Prefix for PLINK binary files (.bed/.bim/.fam)
    sumstats_file : str
        Path to summary statistics file
    output_prefix : str
        Prefix for output files
    clump_p1 : float
        P-value threshold for index variants
    clump_p2 : float
        Secondary significance threshold
    clump_r2 : float
        LD r² threshold
    clump_kb : int
        Physical distance window in kb
    """
    cmd = [
        "plink",
        "--bfile",
        bfile,
        "--clump",
        sumstats_file,
        "--clump-p1",
        str(clump_p1),
        "--clump-p2",
        str(clump_p2),
        "--clump-r2",
        str(clump_r2),
        "--clump-kb",
        str(clump_kb),
        "--out",
        output_prefix,
    ]

    # PLINK writes no .clumped file when no variant reaches clump_p1: drop one left by an earlier run
    Path(f"{output_prefix}.clumped").unlink(missing_ok=True)

    print("Running PLINK clumping...")
    print(f"Command: {' '.join(cmd)}")

    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode != 0:
        print("PLINK stderr:", result.stderr)
        raise RuntimeError(f"PLINK clumping failed with return code {result.returncode}")

    print("Clumping completed successfully")
    return f"{output_prefix}.clumped"


clumped_file = run_plink_clump(
    bfile=plink_prefix,
    sumstats_file=sumstats_file,
    output_prefix="eqtl_clumped",
    clump_p1=clump_p1,
    clump_p2=clump_p2,
    clump_r2=clump_r2,
    clump_kb=clump_kb,
)

Running PLINK clumping...
Command: plink --bfile chr22_genotypes --clump eqtl_sumstats.txt --clump-p1 5e-08 --clump-p2 1e-05 --clump-r2 0.1 --clump-kb 500 --out eqtl_clumped


Clumping completed successfully


### Step 3: Parse and Interpret Clumping Results
PLINK clumping output identifies index SNPs and their associated clumped SNPs. We'll parse this and integrate it with our original results.

In [8]:
def parse_clumped_results(clumped_file):
    """
    Parse PLINK clumped output file.

    Returns
    -------
    pd.DataFrame
        DataFrame with index SNPs and their properties
    """
    clumped = pd.read_csv(clumped_file, sep=r"\s+")

    print(f"Identified {len(clumped)} independent signals")
    print("\nIndex SNPs:")
    print(clumped[["CHR", "SNP", "BP", "P", "TOTAL", "NSIG", "S05", "S01"]].head(10))

    return clumped


clumped_results = parse_clumped_results(clumped_file)

Identified 2 independent signals

Index SNPs:
   CHR                SNP        BP             P  TOTAL  NSIG  S05  S01
0   22  22_42120602_TAA_T  42120602  1.250000e-24   5617  4615   42  153
1   22    22_45393863_T_G  45393863  2.810000e-23    583   292    0  152


### Step 4: Annotate Results with Clumping Information
We'll add clumping information back to our original eQTL results, marking which variants are independent index SNPs.

In [9]:
def annotate_with_clumping(eqtl_results, clumped_results):
    """
    Annotate eQTL results with clumping information.

    Parameters
    ----------
    eqtl_results : pd.DataFrame
        Original eQTL association results
    clumped_results : pd.DataFrame
        PLINK clumped output

    Returns
    -------
    pd.DataFrame
        Annotated results with clumping status
    """
    index_snps = set(clumped_results["SNP"])
    # A SNP is tested for every gene whose window holds it, but PLINK clumps it once, at its
    # smallest P: flag only that gene's row as the index association
    best_row = eqtl_results.dropna(subset=["P"]).groupby("SNP")["P"].idxmin()
    eqtl_results["is_index_snp"] = eqtl_results.index.isin(best_row[best_row.index.isin(index_snps)])

    clumped_mapping = {}
    for _, row in clumped_results.iterrows():
        index_snp = row["SNP"]
        if pd.notna(row.get("SP2")):
            clumped_snps = row["SP2"].split(",")
            for snp in clumped_snps:
                snp = snp.strip()
                if snp and snp != "NONE":  # PLINK 1.9 writes NONE for a clump without other SNPs
                    clumped_mapping[snp.rsplit("(", 1)[0]] = index_snp  # and 'ID(1)' for each SNP

    eqtl_results["index_snp"] = eqtl_results["SNP"].map(
        lambda x: x if x in index_snps else clumped_mapping.get(x, None)
    )

    clump_sizes = clumped_results.set_index("SNP")["TOTAL"].to_dict()
    eqtl_results["clump_size"] = eqtl_results["index_snp"].map(clump_sizes)

    return eqtl_results


annotated_results = annotate_with_clumping(eqtl_results, clumped_results)

print(f"\nTotal associations: {len(annotated_results)}")
print(f"Independent signals (index SNPs): {annotated_results['is_index_snp'].sum()}")
print(f"Associations assigned to clumps: {annotated_results['index_snp'].notna().sum()}")

print("\nIndex SNPs and their associations:")
index_results = annotated_results[annotated_results["is_index_snp"]].sort_values("P")
print(index_results[["SNP", "GENE", "BP", "BETA", "P", "clump_size"]].head(10))


Total associations: 379508
Independent signals (index SNPs): 2


Associations assigned to clumps: 4323

Index SNPs and their associations:
                      SNP             GENE        BP      BETA             P  \
299598  22_42120602_TAA_T  ENSG00000183172  42120602 -0.869484  1.254798e-24   
337170    22_45393863_T_G  ENSG00000100376  45393863  1.118323  2.812677e-23   

        clump_size  
299598      5617.0  
337170       583.0  


## Approach 2: Pre-Analysis LD Pruning
For some analyses, you may want to reduce your variant set before analysis by removing variants in high LD. This is called LD pruning and creates a set of approximately independent variants.

In [10]:
def run_ld_pruning(bfile, output_prefix, window_size=50, step_size=5, r2_threshold=0.5):
    """
    Perform LD pruning to create an independent variant set.

    Parameters
    ----------
    bfile : str
        Prefix for PLINK binary files
    output_prefix : str
        Prefix for output files
    window_size : int
        Window size in variant count
    step_size : int
        Step size for window sliding
    r2_threshold : float
        r² threshold for pruning
    """
    cmd = [
        "plink",
        "--bfile",
        bfile,
        "--indep-pairwise",
        str(window_size),
        str(step_size),
        str(r2_threshold),
        "--out",
        output_prefix,
    ]

    print("Running LD pruning...")
    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode != 0:
        raise RuntimeError(f"LD pruning failed: {result.stderr}")

    prune_in = pd.read_csv(f"{output_prefix}.prune.in", header=None, names=["SNP"])
    print(f"Retained {len(prune_in)} variants after LD pruning")

    return prune_in


pruned_variants = run_ld_pruning(
    bfile=plink_prefix, output_prefix="ld_pruned", window_size=50, step_size=5, r2_threshold=0.5
)

hd_pruned = hd[:, hd.G.var.index.isin(pruned_variants["SNP"])].copy()
print(f"Original variants: {hd.shape[1]}")
print(f"After LD pruning: {hd_pruned.shape[1]}")
print(f"Reduction: {(1 - hd_pruned.shape[1]/hd.shape[1])*100:.1f}%")

Running LD pruning...


Retained 31091 variants after LD pruning


Original variants: 136776
After LD pruning: 31091
Reduction: 77.3%


### Gene-Level Analysis with Clumping
For eQTL analysis, it's often useful to identify independent signals per gene. Here's how to perform gene-stratified clumping.

In [11]:
def clump_per_gene(eqtl_results, bfile, output_dir, clump_p1=0.00000005, clump_p2=0.00001, clump_r2=0.1, clump_kb=500):
    """
    Perform clumping separately for each gene.

    This identifies independent eQTL signals per gene.
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    gene_clumped_results = []

    for gene in eqtl_results["GENE"].unique():
        gene_data = eqtl_results[eqtl_results["GENE"] == gene]

        # PLINK finds no index variant in a gene without any P < clump_p1
        if len(gene_data) < 2 or gene_data["P"].min() >= clump_p1:
            continue

        gene_sumstats = output_dir / f"{gene}_sumstats.txt"
        gene_data[["SNP", "CHR", "BP", "P"]].to_csv(gene_sumstats, sep="\t", index=False)

        gene_clumped = run_plink_clump(
            bfile=bfile,
            sumstats_file=str(gene_sumstats),
            output_prefix=str(output_dir / f"{gene}_clumped"),
            clump_p1=clump_p1,
            clump_p2=clump_p2,
            clump_r2=clump_r2,
            clump_kb=clump_kb,
        )

        try:
            gene_clumped_df = parse_clumped_results(gene_clumped)
            gene_clumped_df["GENE"] = gene
            gene_clumped_results.append(gene_clumped_df)

        except Exception:
            continue

    if not gene_clumped_results:
        print(f"No gene has an index variant with P < {clump_p1}")
        return pd.DataFrame(columns=["CHR", "SNP", "BP", "P", "TOTAL", "GENE"])
    all_gene_clumped = pd.concat(gene_clumped_results, ignore_index=True)

    return all_gene_clumped


gene_clumped = clump_per_gene(
    eqtl_results=eqtl_results,
    bfile=plink_prefix,
    output_dir="gene_clumped",
    clump_p1=clump_p1,
    clump_p2=clump_p2,
    clump_r2=clump_r2,
    clump_kb=clump_kb,
)

print("\nGene-level clumping summary:")
print(gene_clumped.groupby("GENE").size().describe())

Running PLINK clumping...
Command: plink --bfile chr22_genotypes --clump gene_clumped/ENSG00000183172_sumstats.txt --clump-p1 5e-08 --clump-p2 1e-05 --clump-r2 0.1 --clump-kb 500 --out gene_clumped/ENSG00000183172_clumped
Clumping completed successfully
Identified 1 independent signals

Index SNPs:
   CHR                SNP        BP             P  TOTAL  NSIG  S05  S01
0   22  22_42120602_TAA_T  42120602  1.250000e-24    887     3   14   66


Running PLINK clumping...
Command: plink --bfile chr22_genotypes --clump gene_clumped/ENSG00000100376_sumstats.txt --clump-p1 5e-08 --clump-p2 1e-05 --clump-r2 0.1 --clump-kb 500 --out gene_clumped/ENSG00000100376_clumped
Clumping completed successfully
Identified 1 independent signals

Index SNPs:
   CHR              SNP        BP             P  TOTAL  NSIG  S05  S01
0   22  22_45393863_T_G  45393863  2.810000e-23    291     0    0  152



Gene-level clumping summary:
count    2.0
mean     1.0
std      0.0
min      1.0
25%      1.0
50%      1.0
75%      1.0
max      1.0
dtype: float64


### Integration with `cellink` workflows
Finally, let's demonstrate how to integrate clumped results back into the `cellink` ecosystem.

In [12]:
def add_clumping_to_gdata(gdata, clumped_results, key="clumping_info"):
    """
    Add clumping information to the variant annotations in gdata.

    Parameters
    ----------
    gdata : AnnData
        Genotype data object (hd.G)
    clumped_results : pd.DataFrame
        PLINK clumped output
    key : str
        Key to store clumping info in gdata.var
    """
    index_snps = set(clumped_results["SNP"])
    gdata.var[f"{key}_is_index"] = gdata.var.index.isin(index_snps)

    clump_size_map = clumped_results.set_index("SNP")["TOTAL"].to_dict()
    gdata.var[f"{key}_clump_size"] = gdata.var.index.map(clump_size_map)

    nsig_map = clumped_results.set_index("SNP")["NSIG"].to_dict()
    gdata.var[f"{key}_n_sig_secondary"] = gdata.var.index.map(nsig_map)

    print(f"Added clumping annotations to gdata.var with prefix '{key}_'")
    print(f"Index SNPs marked: {gdata.var[f'{key}_is_index'].sum()}")


add_clumping_to_gdata(hd.G, clumped_results, key="eqtl_clump")

hd.G

Added clumping annotations to gdata.var with prefix 'eqtl_clump_'
Index SNPs marked: 2


AnnData object with n_obs × n_vars = 294 × 136776
    obs: 'donor_id', 'sex', 'age'
    var: 'AF', 'AN', 'ER2', 'IMPUTED', 'maf', 'NS', 'R2', 'TYPED', 'TYPED_ONLY', 'id', 'id_mask', 'length', 'pos', 'quality', 'a0', 'a1', 'chrom', 'pos_hg19', 'id_hg19', 'eqtl_clump_is_index', 'eqtl_clump_clump_size', 'eqtl_clump_n_sig_secondary'
    uns: 'dummy_onek1k', 'kinship'
    obsm: 'gPCs', 'PB_CD8 Naive'
    layers: None (.X)